<a href="https://colab.research.google.com/github/nikhilvirdi/Network-Intrusion-Detection-MLP-NSL-KDD/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Environment Check

Before writing any real code, confirm the Colab VM has the libraries we need
and record their exact versions. This matters because Colab's VM is fresh
every session and could, in principle, come with different pre-installed
versions — if something behaves unexpectedly later, version mismatch is the
first thing to rule out.

In [73]:
import sys
import torch
import pandas as pd
import sklearn
import numpy as np

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cpu
pandas: 2.2.2
NumPy: 2.0.2
scikit-learn: 1.6.1


### Step 2: Reproducibility (Random Seed)

Neural network training involves several sources of randomness — most
importantly, how a model's weights are initialized before training starts,
and how data gets shuffled. Without fixing this randomness, running the
exact same code twice can produce two different results, making it
impossible to tell whether a change in outcome came from an actual code
change or just random luck. Setting a fixed seed makes the "randomness"
reproducible — same seed, same sequence of "random" numbers, every run.

In [74]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### Step 3: Download the Dataset

The Colab VM we connected to is a fresh, empty machine — it has no files
on it except what we put there. NSL-KDD isn't bundled with PyTorch or
scikit-learn, so we need to fetch the raw train and test files from an
external source before pandas can read them. We'll download them directly
onto this VM's disk using a shell command, since Colab VM storage is
temporary and resets when the session ends — this download will need to be
redone if this runtime disconnects (e.g. after long inactivity).

In [75]:
!wget -q https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain+.txt
!wget -q https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTest+.txt
!ls -la *.txt

-rw-r--r-- 1 root root  3441513 Aug  3 11:54 KDDTest+.txt
-rw-r--r-- 1 root root 19109424 Aug  3 11:54 KDDTrain+.txt


### Step 4: Load Data into pandas

These two files are plain CSV data, but critically, they have **no header
row** — the very first line is already a real data record, not column
names. If we load this naively, pandas will silently treat that first
record as the header, corrupting the entire dataset without throwing any
error. We'll fix this by explicitly supplying the 41 official NSL-KDD
feature names, plus two extra columns: `label` (the attack type, as text)
and `difficulty_level` (a score of how hard that record is to classify —
metadata about the record, not a real feature, and must never be fed into
the model as input).

In [76]:
column_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

train_df = pd.read_csv('KDDTrain+.txt', names=column_names)
test_df = pd.read_csv('KDDTest+.txt', names=column_names)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (125973, 43)
Test shape: (22544, 43)


### Step 5: Preview the Raw Data

Before deciding how to preprocess anything, we need to see what the data
actually looks like — real values, real types. This tells us which columns
are numeric (something a network can consume directly) versus categorical
text (something that needs converting first), and gives us a first
impression of the data's shape and quality.

In [77]:
train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [78]:
train_df.dtypes

,0
duration,int64
protocol_type,object
service,object
flag,object
src_bytes,int64
dst_bytes,int64
land,int64
wrong_fragment,int64
urgent,int64
hot,int64


### Step 6: Label Distribution

The `label` column holds the actual target we're trying to predict —
but it's not simply "normal" vs "attack." It contains dozens of specific
attack names. Before deciding how to frame this as a classification problem
(binary, or multi-class, or attack categories), we need to see exactly what
values exist and how skewed their counts are — this directly affects our
choice of loss function later.

In [79]:
train_df['label'].value_counts()

,count
label,
normal,67343
neptune,41214
satan,3633
ipsweep,3599
portsweep,2931
smurf,2646
nmap,1493
back,956
teardrop,892


### Step 7: Test Set Label Check

Before finalizing the 5-class framing, we need to confirm two things on
the test set: what labels actually appear there, and specifically which
attack types show up in test but were never seen during training. This is
the concrete generalization gap NSL-KDD is known for — a model that only
memorizes training patterns will struggle precisely on these unseen types.

In [80]:
train_labels = set(train_df['label'].unique())
test_labels = set(test_df['label'].unique())

print("Labels in train:", len(train_labels))
print("Labels in test:", len(test_labels))
print("\nIn test but NOT in train (unseen attacks):")
print(test_labels - train_labels)

Labels in train: 23
Labels in test: 38

In test but NOT in train (unseen attacks):
{'snmpgetattack', 'httptunnel', 'xterm', 'mscan', 'xlock', 'xsnoop', 'sqlattack', 'udpstorm', 'saint', 'processtable', 'snmpguess', 'sendmail', 'mailbomb', 'ps', 'named', 'apache2', 'worm'}


### Step 8: Check `worm` Occurrence Count

Before deciding how to fold `worm` into DoS or R2L, check how many test
rows it actually affects. This tells us whether this ambiguous call
meaningfully impacts evaluation, or is a negligible edge case.

In [81]:
worm_count = (test_df['label'] == 'worm').sum()
print("worm occurrences in test set:", worm_count)
print("as % of test set:", round(worm_count / len(test_df) * 100, 3), "%")

worm occurrences in test set: 2
as % of test set: 0.009 %


### Step 9: Problem Definition

**Task:** Multi-class classification of network connection records.

**Input:** 41 features per connection record (a mix of numeric measurements
and 3 categorical fields: `protocol_type`, `service`, `flag`).

**Output classes (5 total):**
- `normal` — legitimate traffic
- `DoS` (Denial of Service) — e.g. neptune, smurf, back, teardrop, pod,
  land, apache2, mailbomb, processtable, udpstorm, **worm**
- `Probe` — surveillance/scanning, e.g. satan, ipsweep, portsweep, nmap,
  mscan, saint
- `R2L` (Remote-to-Local) — unauthorized remote access, e.g. guess_passwd,
  warezclient, ftp_write, imap, phf, multihop, warezmaster, spy, sendmail,
  named, snmpgetattack, snmpguess, xlock, xsnoop, httptunnel
- `U2R` (User-to-Root) — privilege escalation, e.g. buffer_overflow,
  rootkit, loadmodule, perl, ps, sqlattack, xterm

**Known dataset characteristics to account for later:**
- Severe class imbalance even after 5-class grouping (`normal` and `DoS`
  dominate; `U2R` will remain the smallest by a wide margin) — affects our
  loss function choice (topic 3).
- Test set contains 17 attack types with zero training examples — a real,
  observable generalization gap (topic 4), not a hypothetical one.
- `difficulty_level` column is metadata, excluded from model input entirely.

**Model:** A PyTorch MLP (multi-layer perceptron) trained to take the 41
processed features as input and output a probability distribution over
the 5 classes.

### Step 10: Categorical Feature Inspection

We know `protocol_type`, `service`, and `flag` are text columns that need
converting to numbers before the model can use them. But *how* to convert
them depends on one thing we haven't checked yet: how many distinct values
each one has. A column with 3 possible values needs a different approach
than one with 70 — so we check cardinality (the number of unique values)
before choosing an encoding method.


In [82]:
for col in ['protocol_type', 'service', 'flag']:
    print(col, "->", train_df[col].nunique(), "unique values")

protocol_type -> 3 unique values
service -> 70 unique values
flag -> 11 unique values


### Step 11: Check Categorical Consistency Between Train and Test

Before encoding, verify whether any value in `protocol_type`, `service`,
or `flag` appears in the test set but not in train — the same kind of gap
we already found in the label column. If this exists, encoding train and
test separately would produce mismatched column counts, since one-hot
encoding only creates columns for values it has actually seen.

In [83]:
for col in ['protocol_type', 'service', 'flag']:
    train_vals = set(train_df[col].unique())
    test_vals = set(test_df[col].unique())
    print(col, "-> unseen in test:", test_vals - train_vals)

protocol_type -> unseen in test: set()
service -> unseen in test: set()
flag -> unseen in test: set()


### Step 12: One-Hot Encode Categorical Columns

We'll use scikit-learn's `OneHotEncoder`, which follows a two-stage
pattern used throughout sklearn: **fit**, then **transform**.

- `fit` means "look at this data and learn what categories exist" —
  it doesn't change any data, it just records the vocabulary.
- `transform` means "apply that already-learned encoding to this data."

Critically, we `fit` **only on the training set**, then use that same
learned encoding to `transform` both train and test. We do NOT let the
encoder look at test data while learning categories. Right now this
doesn't change the outcome, since we already confirmed test has no
categories missing from train — but it's a discipline worth building now,
because it becomes essential later with numeric scaling, where fitting on
test data would leak information about the test set into training and
quietly inflate our results. Better to build the habit here, where the
stakes are low, than get it wrong later where they aren't.

In [84]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['protocol_type', 'service', 'flag']

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(train_df[cat_cols])

train_encoded = encoder.transform(train_df[cat_cols])
test_encoded = encoder.transform(test_df[cat_cols])

print("Train encoded shape:", train_encoded.shape)
print("Test encoded shape:", test_encoded.shape)

Train encoded shape: (125973, 84)
Test encoded shape: (22544, 84)


### Step 13: Numeric Feature Inspection

Before deciding how to scale the numeric features, we need to see their
actual ranges. Neural networks are sensitive to the *scale* of input
numbers — a feature ranging into the tens of thousands sitting next to
one that's always 0 or 1 can cause problems during training (more on
exactly why when we reach the training loop, topic 3). First, confirm
this scale mismatch is real in our data, not assumed.

In [85]:
numeric_cols = [col for col in train_df.columns if col not in ['protocol_type', 'service', 'flag', 'label', 'difficulty_level']]
train_df[numeric_cols].describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
count,125973.00000,1.259730e+05,1.259730e+05,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,...,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000
mean,287.14465,4.556674e+04,1.977911e+04,0.000198,0.022687,0.000111,0.204409,0.001222,0.395736,0.279250,...,182.148945,115.653005,0.521242,0.082951,0.148379,0.032542,0.284452,0.278485,0.118832,0.120240
std,2604.51531,5.870331e+06,4.021269e+06,0.014086,0.253530,0.014366,2.149968,0.045239,0.489010,23.942042,...,99.206213,110.702741,0.448949,0.188922,0.308997,0.112564,0.444784,0.445669,0.306557,0.319459
min,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,82.000000,10.000000,0.050000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.00000,4.400000e+01,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,255.000000,63.000000,0.510000,0.020000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.00000,2.760000e+02,5.160000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,255.000000,255.000000,1.000000,0.070000,0.060000,0.020000,1.000000,1.000000,0.000000,0.000000
max,42908.00000,1.379964e+09,1.309937e+09,1.000000,3.000000,3.000000,77.000000,5.000000,1.000000,7479.000000,...,255.000000,255.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Step 14: Scale Numeric Features

Using scikit-learn's `StandardScaler`, following the same fit/transform
discipline as before: **fit only on training data**, then apply that
same learned scaling to both train and test. This time, fitting on test
data would be a real problem, not just a discipline exercise — if the
scaler learned the mean/std from test data too, information about the
test set would leak into how we scale the training data, quietly making
our model's evaluation look better than it truly would be on genuinely
unseen data.

In [86]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(train_df[numeric_cols])

train_scaled = scaler.transform(train_df[numeric_cols])
test_scaled = scaler.transform(test_df[numeric_cols])

print("Train scaled shape:", train_scaled.shape)
print("Test scaled shape:", test_scaled.shape)
print("\nMean of first column (should be ~0):", train_scaled[:, 0].mean())
print("Std of first column (should be ~1):", train_scaled[:, 0].std())

Train scaled shape: (125973, 38)
Test scaled shape: (22544, 38)

Mean of first column (should be ~0): 2.549477400423625e-17
Std of first column (should be ~1): 1.0


### Step 15: Combine Numeric and Categorical Features

We now have two separate NumPy arrays per dataset split — `train_scaled`
(38 numeric columns) and `train_encoded` (84 one-hot categorical columns),
same for test. These need to be joined side-by-side into one array per
split, since a model expects a single input matrix, not several separate
pieces. The final feature count should be 38 + 84 = 122 columns.

In [87]:
import numpy as np

X_train = np.hstack([train_scaled, train_encoded])
X_test = np.hstack([test_scaled, test_encoded])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (125973, 122)
X_test shape: (22544, 122)


### Progress Summary (Steps 1–15)

**Step 1 — Environment Check:** Confirmed the Colab machine has all the
tools we need (PyTorch, pandas, NumPy, scikit-learn) and recorded their
exact versions, so we have a reference point if anything behaves oddly later.

**Step 2 — Reproducibility (Random Seed):** Fixed a "seed" number so that
any randomness in our process (like how a model starts learning) happens
the same way every time we re-run the notebook — letting us trust that
changes in results come from our actual changes, not random luck.

**Step 3 — Download the Dataset:** Pulled the two official NSL-KDD data
files (one for training, one for testing) onto this Colab machine.

**Step 4 — Load Data into pandas:** Read those files into table form,
manually attaching the correct column names, since the raw files don't
include headers on their own.

**Step 5 — Preview the Raw Data:** Looked at actual rows and column types,
confirming which of the 41 features are numbers and which are text.

**Step 6 — Label Distribution:** Checked how many times each attack type
appears in training. Found the data is heavily lopsided — a couple of
attack types make up the vast majority of records, while several others
appear only a handful of times.

**Step 7 — Test Set Label Check:** Compared training labels against test
labels, and found the test set contains 17 attack types that never once
appear in training — the real, built-in challenge of this dataset.

**Step 8 — Check `worm` Occurrence:** Investigated one borderline attack
type (`worm`) that different sources classify differently. Found it barely
appears in test data at all (2 records), so we assigned it to DoS and moved
on without over-debating a near-irrelevant edge case.

**Step 9 — Problem Definition:** Formally wrote down what we're building:
a model that classifies each connection as `normal` or one of 4 attack
categories (`DoS`, `Probe`, `R2L`, `U2R`) — 5 classes total.

**Step 10 — Categorical Feature Inspection:** Checked how many distinct
values the 3 text columns (`protocol_type`, `service`, `flag`) contain,
to decide how to convert them into numbers.

**Step 11 — Categorical Consistency Check:** Confirmed every text value in
the test set also appears in training — no surprises there, unlike the
label column.

**Step 12 — One-Hot Encode Categorical Columns:** Converted the 3 text
columns into 84 separate yes/no columns (one per possible value), since a
model can only process numbers, not words like "tcp" or "http."

**Step 13 — Numeric Feature Inspection:** Looked at the actual ranges of
the 38 numeric features, and found some values reach into the billions
while others are capped at 0 or 1 — a scale mismatch that needs fixing.

**Step 14 — Scale Numeric Features:** Rescaled all 38 numeric features
onto a comparable footing, so no single feature dominates the model's
calculations just because its raw numbers happen to be bigger.

**Step 15 — Combine Numeric and Categorical Features:** Joined the 38
rescaled numeric columns and the 84 yes/no categorical columns into one
final table — 122 numbers per connection record. This is the actual input
the model will receive, saved as `X_train` and `X_test`.
